# 02 · Match the DFL data to the challenge tracking

The DFL tracking is corrected: ball and players are never missing. In the automatic challenge tracking the ball is often missing, especially during stoppages. A model trained on clean data would learn "ball visible = ball in play". We therefore degrade the DFL data until its statistics resemble those of the challenge.

Only **aggregates of the challenge input** are matched (dropout rate, run lengths, ball acceleration, distance to the nearest player), never labels.

**Output:** `artifacts/dfl_d67s/<match>.parquet`, training data for notebook 04.

In [1]:
# Standard library
import itertools
import xml.etree.ElementTree as ET
from pathlib import Path

# Data
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent
DFL = ROOT / "artifacts" / "dfl"
OUT = ROOT / "artifacts" / "dfl_d67s"
OUT.mkdir(parents=True, exist_ok=True)
ATD_RAW = ROOT / "tactical" / "BAR-ATH_tactical_atd.txt"
ATD_META = ROOT / "tactical" / "BAR-ATH_tactical_atd.xml"

FPS = 25
N_FRAMES = 199_319
PITCH = np.array([105.3, 68.0])  # games.py, measured from corrected tracking
HALVES = [(11185, 90651), (110808, 190197)]  # DATA.md, inclusive
RANDOM_STATE = 42
M_DEAD = 0.67  # ball dropout rate during stoppages (scenario d67), fixed assumption
MAX_BALL_SPEED = 40.0  # m/s, faster is physically implausible → missing
MAX_PLAYER_SPEED = 12.0  # m/s, faster means a track jump → missing
S_GRID, SIGMA_B_GRID, OFFSET_CORR_S = [0, 2, 4, 6, 8], [0.0, 0.5, 1.0, 1.5], 1.0

## 1. Load the challenge tracking and DFL

The challenge format follows the EPTS specification in the XML: one line per frame, player slots as `x,y,s`, then the ball. Normalised coordinates are converted to metres with the centre spot at (0, 0). Only the two graded halves are used.

In [2]:
meta = ET.parse(ATD_META).getroot()
team_side = {t.get("id"): s for t, s in zip(meta.findall("./Metadata/Teams/Team"), "HA")}
player_team = {p.get("id"): team_side[p.get("teamId")] for p in meta.findall("./Metadata/Players/Player")}
channel_player = {c.get("id"): c.get("playerId") for c in meta.findall("./Metadata/PlayerChannels/PlayerChannel")}
spec = meta.find("./DataFormatSpecifications/DataFormatSpecification")
slot_side = np.array([player_team[channel_player[g.find("PlayerChannelRef").get("playerChannelId")]]
                      for g in spec.findall("./SplitRegister/SplitRegister")])

xy = np.empty((N_FRAMES, len(slot_side) + 1, 2), dtype=np.float32)
with ATD_RAW.open() as fh:
    for i, line in enumerate(fh):
        _, players, ball = line.strip().split(":")
        xy[i] = np.fromstring(players.replace(";", ",") + "," + ball, sep=",").reshape(-1, 3)[:, :2]
xy = (xy - 0.5) * PITCH

graded, period = np.zeros(N_FRAMES, bool), np.zeros(N_FRAMES, int)
for k, (a, b) in enumerate(HALVES, 1):
    graded[a:b + 1], period[a:b + 1] = True, k
ch = {"ball": xy[graded, -1], "players": xy[graded, :-1], "period": period[graded], "side": slot_side}

def dfl_arrays(path):
    df = pd.read_parquet(path)
    cols = [c[:-2] for c in df.columns if c.endswith("_x") and c[:2] in ("H_", "A_")]
    players = np.stack([df[[f"{c}_x" for c in cols]].to_numpy(), df[[f"{c}_y" for c in cols]].to_numpy()], axis=-1)
    return {"ball": df[["ball_x", "ball_y"]].to_numpy(np.float32), "players": players.astype(np.float32),
            "period": df["period"].to_numpy(), "side": np.array([c[0] for c in cols]), "label": df["label"].to_numpy()}

dfl = {p.stem: dfl_arrays(p) for p in sorted(DFL.glob("J03W*.parquet"))}
print(f"Challenge: {graded.sum():,} graded frames · DFL: {len(dfl)} matches")

Challenge: 158,857 graded frames · DFL: 7 matches


## 2. Plausibility rule and dropout statistics

Both domains get the same preprocessing: positions that would require a jump above 40 m/s (ball) or 12 m/s (players) are treated as missing. We then measure on the challenge data how often and how long ball and players are missing.

In [3]:
def clean(d):
    d = dict(d)
    ball, players = d["ball"].copy(), d["players"].copy()
    ball[np.r_[0, np.linalg.norm(np.diff(ball, axis=0), axis=1) * FPS] > MAX_BALL_SPEED] = np.nan
    step = np.linalg.norm(np.diff(players, axis=0), axis=-1) * FPS
    players[np.r_[np.zeros((1, players.shape[1])), step] > MAX_PLAYER_SPEED] = np.nan
    d["ball"], d["players"] = ball, players
    return d

def runs(mask, period):
    """Lengths [frames] of True runs, per half."""
    out = []
    for p in np.unique(period):
        m = np.r_[False, mask[period == p], False].astype(int)
        d = np.flatnonzero(np.diff(m))
        out.append(d[1::2] - d[::2])
    return np.concatenate(out)

chc = clean(ch)
seen_c = np.isfinite(chc["ball"]).all(1)
G = runs(~seen_c, chc["period"])
S = runs(seen_c, chc["period"])
P = np.concatenate([runs(np.isfinite(chc["players"][:, j]).all(-1), chc["period"]) for j in range(chc["players"].shape[1])])

M_BALL = 1 - seen_c.mean()
M_PLAYER = 1 - np.isfinite(chc["players"]).all(-1).sum(1).mean() / 22
MU_ABSENT = P.mean() * M_PLAYER / (1 - M_PLAYER)
PI_D = np.median([(d["label"] == "D").mean() for d in dfl.values()])
M_LIVE = (M_BALL - PI_D * M_DEAD) / (1 - PI_D)  # so that the overall dropout rate matches the challenge

pd.Series({"ball dropout challenge": M_BALL, "gap mean [s]": G.mean() / FPS, "seen run mean [s]": S.mean() / FPS,
           "player dropout": M_PLAYER, "slot run mean [s]": P.mean() / FPS, "D share DFL": PI_D,
           "ball dropout in D": M_DEAD, "ball dropout live": M_LIVE}).round(3)

ball dropout challenge      0.408
gap mean [s]               15.723
seen run mean [s]          22.926
player dropout              0.069
slot run mean [s]         100.711
D share DFL                 0.443
ball dropout in D           0.670
ball dropout live           0.200
dtype: float64

## 3. Generate dropouts

**Ball:** a Markov process with the states "seen" and "missing". Gaps end at the same rate in both game states (mean challenge gap). Seen runs end more often during stoppages, so that 67 % of stoppage frames have no ball, and in open play just enough that the overall rate matches the challenge.

**Players:** per slot, alternating missing and seen blocks with lengths taken from the challenge.

In [4]:
def renewal(n, m, rng, draw_gap, draw_seen):
    """Alternating missing/seen blocks; True = missing. Starts missing with probability m."""
    out, pos, missing, first = np.zeros(n, bool), 0, rng.random() < m, True
    while pos < n:
        L = int(draw_gap() if missing else draw_seen())
        L = int(rng.integers(1, L + 1)) if first else L
        out[pos:pos + L], pos, missing, first = missing, pos + L, not missing, False
    return out

def degrade(d, m_dead, m_live, seed):
    rng = np.random.default_rng(seed)
    dead, per, n = d["label"] == "D", d["period"], len(d["label"])

    m_z = {True: m_dead, False: m_live}
    seen_mean = {z: G.mean() * (1 - m) / m for z, m in m_z.items()}
    change = np.flatnonzero(np.r_[True, (dead[1:] != dead[:-1]) | (per[1:] != per[:-1]), True])
    half_start = set(np.flatnonzero(np.r_[False, per[1:] != per[:-1]]))
    next_change = change[np.searchsorted(change, np.arange(n), side="right")]

    miss = np.zeros(n, bool)
    pos, missing = 0, rng.random() < m_z[dead[0]]
    while pos < n:
        if pos in half_start:
            missing = rng.random() < m_z[dead[pos]]
        L = rng.geometric(1 / (G.mean() if missing else seen_mean[dead[pos]]))
        end = min(pos + L, next_change[pos])
        miss[pos:end] = missing
        missing = missing if pos + L > end else not missing  # cut at a state boundary → block continues with the new rate
        pos = end

    ball, players = d["ball"].copy(), d["players"].copy()
    ball[miss] = np.nan
    for j in range(players.shape[1]):
        absent = renewal(n, M_PLAYER, rng, lambda: max(1, round(rng.exponential(MU_ABSENT))), lambda: rng.choice(P))
        players[absent, j] = np.nan
    return clean({**d, "ball": ball, "players": players})

# Seeds as in development (d67 was the second of three scenarios there), so the data stays bit-identical.
d67 = {mid: degrade(d, M_DEAD, M_LIVE, RANDOM_STATE + 1000 + mi) for mi, (mid, d) in enumerate(dfl.items())}

## 4. Ball smoothing and ball offset

The challenge ball is smoothed and lies further from the players than the DFL ball. On a small grid we choose a smoothing `s` (frames) and a slowly drifting offset `sigma_b` (m) so that ball acceleration near a player and distance to the nearest player (median, 90th percentile) come as close as possible to the challenge.

In [5]:
def gauss_kernel(sigma):
    r = int(np.ceil(3 * sigma))
    k = np.exp(-0.5 * (np.arange(-r, r + 1) / sigma) ** 2)
    return k / k.sum()

def smooth_observed(x, period, sigma):
    if sigma == 0:
        return x.copy()
    out, k = np.full_like(x, np.nan), gauss_kernel(sigma)
    for p in np.unique(period):
        idx = np.flatnonzero(period == p)
        w = np.isfinite(x[idx]).all(1).astype(float)
        den = np.convolve(w, k, mode="same")
        for j in range(x.shape[1]):
            num = np.convolve(np.nan_to_num(x[idx, j]) * w, k, mode="same")
            out[idx, j] = np.where(w > 0, num / np.maximum(den, 1e-9), np.nan)
    return out

def ball_transform(ball, period, s, sigma_b, rng):
    b = ball.copy()
    if sigma_b > 0:
        noise = np.stack([np.convolve(rng.standard_normal(len(b)), gauss_kernel(OFFSET_CORR_S * FPS), mode="same") for _ in range(2)], axis=1)
        b = b + noise / noise.std(0) * sigma_b
    return smooth_observed(b, period, s)

def ball_stats(ball, players):
    seen = np.isfinite(ball).all(1)
    dist = np.linalg.norm(players - ball[:, None], axis=-1)
    dn = np.where(np.isfinite(dist), dist, np.inf).min(1)
    h = 3
    v = np.full_like(ball, np.nan)
    v[h:-h] = (ball[2 * h:] - ball[:-2 * h]) / (2 * h / FPS)
    acc = np.full(len(ball), np.nan)
    acc[h:-h] = np.linalg.norm(v[2 * h:] - v[:-2 * h], axis=1) / (2 * h / FPS)
    a = acc[seen & (dn <= 3) & np.isfinite(acc)]
    dd = dn[seen & np.isfinite(dn)]
    return {"acc_q50": np.quantile(a, 0.5), "acc_q90": np.quantile(a, 0.9), "dist_q50": np.quantile(dd, 0.5), "dist_q90": np.quantile(dd, 0.9)}

target = pd.Series(ball_stats(chc["ball"], chc["players"]))
grid = []
for s, sb in itertools.product(S_GRID, SIGMA_B_GRID):
    st = pd.DataFrame([ball_stats(ball_transform(d["ball"], d["period"], s, sb, np.random.default_rng(RANDOM_STATE + 7)), d["players"])
                       for d in d67.values()]).median()
    grid.append({"s": s, "sigma_b": sb, **st, "deviation": np.mean(np.abs(np.log(st / target)))})
grid = pd.DataFrame(grid)
best = grid.loc[grid["deviation"].idxmin()]
S_BEST, SIGMA_B_BEST = int(best["s"]), float(best["sigma_b"])

print(f"Chosen: s = {S_BEST} frames, sigma_b = {SIGMA_B_BEST} m")
grid.sort_values("deviation").head(5).round(3)

Chosen: s = 6 frames, sigma_b = 0.5 m


,s,sigma_b,acc_q50,acc_q90,dist_q50,dist_q90,deviation
13,6,0.5,3.680,17.377,2.483,8.579,0.224
12,6,0.0,3.589,17.354,2.435,8.652,0.228
14,6,1.0,3.727,17.116,2.708,8.547,0.245
15,6,1.5,3.901,17.221,3.003,8.675,0.262
16,8,0.0,3.597,13.792,2.505,8.642,0.292


## 5. Save and check

In [6]:
def save(mid, dd):
    df = pd.read_parquet(DFL / f"{mid}.parquet")
    cols = [c[:-2] for c in df.columns if c.endswith("_x") and c[:2] in ("H_", "A_")]
    df[["ball_x", "ball_y"]] = dd["ball"]
    df.loc[~np.isfinite(dd["ball"]).all(1), "ball_z"] = np.nan
    df[[f"{c}_x" for c in cols]], df[[f"{c}_y" for c in cols]] = dd["players"][..., 0], dd["players"][..., 1]
    df.to_parquet(OUT / f"{mid}.parquet")

d67s = {}
for mi, (mid, dd) in enumerate(d67.items()):
    b2 = ball_transform(dd["ball"], dd["period"], S_BEST, SIGMA_B_BEST, np.random.default_rng(RANDOM_STATE + 6000 + mi))
    d67s[mid] = {**dd, "ball": clean({**dd, "ball": b2})["ball"]}
    save(mid, d67s[mid])

pd.DataFrame({
    "Challenge": {**target, "ball seen": seen_c.mean()},
    "DFL d67s (median)": pd.DataFrame([{**ball_stats(d["ball"], d["players"]), "ball seen": np.isfinite(d["ball"]).all(1).mean()}
                                       for d in d67s.values()]).median(),
}).T.round(3)

,acc_q50,acc_q90,dist_q50,dist_q90,ball seen
Challenge,5.631,17.430,1.908,6.993,0.592
DFL d67s (median),3.615,17.141,2.501,8.581,0.562


**Result:** the degraded DFL data (`d67s`) is close to the challenge in ball visibility, ball acceleration and player distance. These seven matches, together with the Metrica matches (notebook 03), are the training data.